# E-Commerce Orders — Data Cleaning & Preparation

**Dataset:** Raw order log — 315 rows, 10 columns  
**Issues identified:** duplicates, mixed date formats, category inconsistencies, nulls, invalid values

> This notebook covers the first stage of the analysis pipeline: inspecting and cleaning the raw data.  
> The cleaned dataset is then exported to Excel and loaded into Google Sheets for aggregation, 
> statistical analysis and dashboard visualisation.

→ **[Google Sheets Dashboard](https://docs.google.com/spreadsheets/d/1qmkZ7p7UZYsG6Zz2rcFbURAcyFC-bU67qrH0tbrMhLw/edit?usp=sharing)** — metrics, charts and key insights

## Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


## 1. Load Data

In [3]:
df = pd.read_csv('ecommerce_raw.csv')
print(f"Shape: {df.shape}")
df.sample(5)


Shape: (315, 10)


,order_id,customer_id,order_date,category,product,unit_price,quantity,country,payment_method,status
31,176,CUST1013,2024-02-16,Sports,Tent,90.73,4.00,UK,Bank Transfer,Pending
112,278,CUST1290,2024-09-27,Home & Garden,Rug,195.69,1.00,USA,PayPal,Completed
223,46,CUST1160,2024-01-20,Home & Garden,Plant Pot,120.31,3.00,USA,Bank Transfer,Completed
285,116,CUST1101,2024-12-19,Books,Self-Help,28.45,4.00,Canada,Bank Transfer,Pending
158,41,CUST1286,15/12/2024,Home & Garden,Lamp,180.80,2.00,UK,Credit Card,Completed


## 2. Diagnosis

In [4]:
print("── Nulls ──────────────────────────")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string())

print(f"\n── Duplicates ─────────────────────")
print(f"{df.duplicated().sum()} rows")

print(f"\n── Categories (raw) ───────────────")
print(sorted(df['category'].unique()))

print(f"\n── Date samples ───────────────────")
print(df['order_date'].unique()[:12])

print(f"\n── Invalid values ─────────────────")
print(f"Negative prices : {(df['unit_price'] < 0).sum()}")
print(f"Zero quantities : {(df['quantity'] == 0).sum()}")


── Nulls ──────────────────────────
customer_id        9
unit_price        15
quantity          12
country           12
payment_method     9

── Duplicates ─────────────────────
3 rows

── Categories (raw) ───────────────
['Bokks', 'Books', 'CLOTHING', 'Clothing', 'Clothng', 'ELECTRONICS', 'Electronics', 'Electronicss', 'Home & Garden', 'Home and Garden', 'Home&Garden', 'SPORTS', 'Sports', 'Sprots', 'books', 'clothing', 'electronics', 'home & garden', 'sports']

── Date samples ───────────────────
['2024-05-25' '2024-08-24' '2024-08-21' 'Nov 24 2024' '2024-07-22'
 '2024-04-28' '2024-12-24' '2024-02-17' '2024-03-21' '2024-10-25'
 '2024-03-02' '2024-03-04']

── Invalid values ─────────────────
Negative prices : 8
Zero quantities : 6


## 3. Cleaning

### 3.1 Duplicates

In [5]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed: {before - len(df)} rows → {len(df)} remaining")


Removed: 3 rows → 312 remaining


### 3.2 Date Format

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=False)
df['year']  = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%b')
print("Unified to datetime. Sample:")
print(df['order_date'].head(5).to_string())


Unified to datetime. Sample:
0   2024-05-25
1   2024-08-24
2   2024-08-21
3   2024-11-24
4   2024-07-22


### 3.3 Category Normalisation

In [7]:
mapping = {
    'electronics': 'Electronics', 'electronicss': 'Electronics',
    'clothing': 'Clothing',       'clothng': 'Clothing',
    'sports': 'Sports',           'sprots': 'Sports',
    'books': 'Books',             'bokks': 'Books',
    'home & garden': 'Home & Garden',
    'home and garden': 'Home & Garden',
    'home&garden': 'Home & Garden',
}
df['category'] = df['category'].str.strip().str.lower().replace(mapping)
print(f"Unique categories after: {sorted(df['category'].unique())}")


Unique categories after: ['Books', 'Clothing', 'Electronics', 'Home & Garden', 'Sports']


### 3.4 Whitespace

In [8]:
str_cols = ['product', 'country', 'payment_method', 'status', 'customer_id']
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().replace('nan', np.nan)


### 3.5 Invalid Values

In [9]:
before = len(df)
df = df[(df['unit_price'] > 0) & (df['quantity'] > 0)]
print(f"Dropped {before - len(df)} rows with negative price or zero quantity")


Dropped 40 rows with negative price or zero quantity


### 3.6 Missing Values

In [10]:
# unit_price → median by category
df['unit_price'] = df.groupby('category')['unit_price'].transform(
    lambda x: x.fillna(x.median())
)
# quantity → 1
df['quantity'] = df['quantity'].fillna(1).astype(int)
# country, payment_method → mode
for col in ['country', 'payment_method']:
    df[col] = df[col].fillna(df[col].mode()[0])
# customer_id → 'Unknown'
df['customer_id'] = df['customer_id'].fillna('Unknown')

print(f"Remaining nulls: {df.isnull().sum().sum()}")


Remaining nulls: 0


### 3.7 Add `total_revenue`

In [11]:
df['total_revenue'] = (df['unit_price'] * df['quantity']).round(2)


## 4. Validation

In [12]:
assert df.isnull().sum().sum() == 0,  "Nulls remain"
assert (df['unit_price'] > 0).all(),  "Negative prices remain"
assert (df['quantity']   > 0).all(),  "Zero quantities remain"
assert df['category'].nunique() == 5, "Unexpected categories"

print("All checks passed.")
print(f"Final shape : {df.shape}")
print(f"\n{df.dtypes}")
df.head()


All checks passed.
Final shape : (272, 14)

order_id                   int64
customer_id               object
order_date        datetime64[ns]
category                  object
product                   object
unit_price               float64
quantity                   int64
country                   object
payment_method            object
status                    object
year                       int32
month                      int32
month_name                object
total_revenue            float64
dtype: object


,order_id,customer_id,order_date,category,product,unit_price,quantity,country,payment_method,status,year,month,month_name,total_revenue
1,162,CUST1292,2024-08-24,Electronics,Camera,745.37,4,Germany,Credit Card,Completed,2024,8,Aug,2981.48
2,272,CUST1247,2024-08-21,Electronics,Smartphone,90.24,5,Canada,Credit Card,Completed,2024,8,Aug,451.20
3,202,CUST1264,2024-11-24,Books,Fiction Novel,12.01,3,USA,Bank Transfer,Completed,2024,11,Nov,36.03
4,187,CUST1097,2024-07-22,Electronics,Headphones,1125.38,5,Germany,Bank Transfer,Completed,2024,7,Jul,5626.90
5,60,CUST1032,2024-04-28,Home & Garden,Rug,25.31,2,Australia,PayPal,Returned,2024,4,Apr,50.62


## 5. Export

In [14]:
df.to_excel('ecommerce_clean.xlsx', index=False)
print("Saved: ecommerce_clean.xlsx")
print(f"Rows: {len(df)}  |  Columns: {list(df.columns)}")


Saved: ecommerce_clean.xlsx
Rows: 272  |  Columns: ['order_id', 'customer_id', 'order_date', 'category', 'product', 'unit_price', 'quantity', 'country', 'payment_method', 'status', 'year', 'month', 'month_name', 'total_revenue']


---
**Next step:** statistics, aggregations and dashboard visualisation  
→ [Google Sheets Dashboard](https://docs.google.com/spreadsheets/d/1qmkZ7p7UZYsG6Zz2rcFbURAcyFC-bU67qrH0tbrMhLw/edit?usp=sharing) 